# Parsing with an LLM

**Module 2 — Lesson 9**

Both the regex parser and GLiNER are free, fast, and deterministic. But neither handles garbled OCR output, novel layouts, or emails where the field boundaries have completely collapsed. An LLM handles what neither can — through comprehension rather than pattern matching.

In this notebook you will:

- Design a parsing prompt that returns predictable, parseable output
- See how an LLM and the template parser produce identical results on clean text — and very different results on noisy text
- Use the Batch API to process failures in bulk at reduced cost
- Merge the LLM results with the template results into a single dataset


First, install dependencies.

In [1]:
%pip install openai python-dotenv tiktoken -q


Note: you may need to restart the kernel to use updated packages.


Next, import packages, load your env variables and define the LLM.

In [2]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path=".env")

if not os.environ.get("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY not found. Create a .env file with your key:\n"
        "  OPENAI_API_KEY=sk-..."
    )

client = OpenAI()

TXT_DIR = Path("data/extracted_text")

Load the parsing failures from the earlier notebook.


In [3]:
failure_list_path = Path("data/parse_failures_enron.json")

with open(failure_list_path) as f:
    failure_ids = json.load(f)

failure_paths = [TXT_DIR / f"{doc_id}.txt" for doc_id in failure_ids]
print(f"{len(failure_paths)} failures loaded")


70 failures loaded


## 1. Prompt design

The prompt below asks the LLM to return one field per line in a fixed format — a field name followed by an array of values. This makes the response predictable and easy to parse on our end. The LLM's only job is to identify which values belong to which field. We handle the structuring.

The example in the prompt shows the LLM exactly what correct output looks like, including how to handle OCR-corrupted field labels and redacted content.


In [4]:
PARSE_PROMPT = """Given this email text, extract the header fields.

Return one field per line in this exact format:

SENDER_NAME: ["Mary Cook"]
SENDER_EMAIL: ["mary.cook@enron.com"]
RECIPIENT_NAMES: ["Stephanie Panus", "Susan Bailey", "[REDACTED] B6"]
RECIPIENT_EMAILS: []
CC_NAMES: ["Tana Jones"]
CC_EMAILS: ["tana.jones@enron.com"]
DATE: ["Wednesday, May 09, 2001 02:52 AM"]
SUBJECT: ["Paralegal Lunch"]

"Toa:" is OCR for "To:", "Ce:" is "Cc:", "Bate:" is "Date:". Correct field labels, not values.
Extract every value as it appears, including [REDACTED] markers.
Only extract from the first email — ignore forwarded chains.
Ignore government stamps (CONFIDENTIAL, Case No, Doc No, ENRON CORP, etc.).
Use empty arrays [] for fields not present in the text.

Email text:
{text}"""


## 2. Single-call parsing

Before sending all 70 failures through the Batch API, test the prompt on a single email to check that the response format is correct and the LLM is extracting the right fields.

The next two cells define the parsing functions and then test them on the first failure email. Compare the raw text to the extracted fields — the LLM should handle the OCR corruption that the template parser couldn't.

In [5]:
import ast

def parse_llm_response(content):
    """Parse an LLM text response into a dict of arrays."""
    parsed = {}
    for line in content.strip().split("\n"):
        if ": [" in line:
            key, val = line.split(": ", 1)
            try:
                parsed[key.strip()] = ast.literal_eval(val.strip())
            except (ValueError, SyntaxError):
                parsed[key.strip()] = []
    return parsed


def parse_with_llm(text, client):
    """Send an email to the LLM and parse the response."""
    response = client.chat.completions.create(
        model="gpt-4.1-2025-04-14",
        messages=[{
            "role": "user",
            "content": PARSE_PROMPT.format(text=text[:2000])
        }],
    )
    return parse_llm_response(response.choices[0].message.content)


The two functions above handle the API call and response parsing. `parse_llm_response` uses `ast.literal_eval` to parse the arrays -- if the LLM returns malformed syntax, the `try/except` silently produces an empty list for that field rather than crashing. This is intentional: bad responses produce incomplete records, not errors.

The cell below tests them on the first failure email.

In [6]:
sample_text = failure_paths[0].read_text(encoding="utf-8")
print("Raw text (first 500 chars):")
print(sample_text[:500])
print()
result = parse_with_llm(sample_text, client)
print("LLM result:")
for key, values in result.items():
    print(f"  {key}: {values}")


Raw text (first 500 chars):
CONFIDENTIAL
Enron
Corp.
Case
No.
EC-2002-01038
Doc
No.
E076559BC
Date:
01/15/2003
ENRON
CORP.
-
PRODUCED
PURSUANT
TO
FERC
SUBPOENA.
SUBJECT
TO PROTECTIVE
ORDER.
CONFIDENTIAL
TREATMENT REQUESTED.
RELEASE
IN
FULL
From:
Mary Hain <mary.hain@enron.com>
Sent:
Tue,
30
Jan 2001
00:42:00
-0800
(PST)
To:
Phillip K Allen <phillip.allen@enron.com>,
Robert Badeer
<robert.badeer@enron.com>,
Tim Belden <tim.belden@enron.com>,
Shelia
Benke <shelia.benke@enron.com>,
Donald M- ECT Origination Black
<donald.blac

LLM result:
  SENDER_NAME: ['Mary Hain']
  SENDER_EMAIL: ['mary.hain@enron.com']
  RECIPIENT_NAMES: ['Phillip K Allen', 'Robert Badeer', 'Tim Belden', 'Shelia Benke', 'Donald M- ECT Origination Black', 'William S Bradford', 'Rick Buy', 'Andre Cangucu', 'Alan Comnes', 'Wanda Curry', 'Jeff Dasovich', 'Karen Denne']
  RECIPIENT_EMAILS: ['phillip.allen@enron.com', 'robert.badeer@enron.com', 'tim.belden@enron.com', 'shelia.benke@enron.com', 'donald.black@enron.com', 'wil

### Optional: numbered encoding for token savings

The section below is an optional technique -- skip ahead to section 3 if you want to continue with the main pipeline.

Rather than sending the raw email text and asking the LLM to return the values, you could number every word in the email and ask it to return the numbers instead. The LLM works with a compact codebook, and you decode the numbers back to words on your end.

This reduces output tokens -- a number like `42` is one token, regardless of how long the word it represents is. At scale, this adds up.

In [7]:
def encode_email(text):
    """Replace each word with a number, building a codebook."""
    codebook = {}
    encoded_words = []
    
    for idx, word in enumerate(text.split()):
        tag = str(idx)
        codebook[tag] = word
        encoded_words.append(tag)
    
    return encoded_words, codebook


sample = failure_paths[0].read_text(encoding="utf-8")
encoded, codebook = encode_email(sample)

print(f"{len(codebook)} words encoded")
print()
print("Codebook (first 20):")
for tag, word in list(codebook.items())[:20]:
    print(f"  {tag} = {word}")
print()
print("Encoded email (first 40):")
print(" ".join(encoded[:40]))


483 words encoded

Codebook (first 20):
  0 = CONFIDENTIAL
  1 = Enron
  2 = Corp.
  3 = Case
  4 = No.
  5 = EC-2002-01038
  6 = Doc
  7 = No.
  8 = E076559BC
  9 = Date:
  10 = 01/15/2003
  11 = ENRON
  12 = CORP.
  13 = -
  14 = PRODUCED
  15 = PURSUANT
  16 = TO
  17 = FERC
  18 = SUBPOENA.
  19 = SUBJECT

Encoded email (first 40):
0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39


Now the prompt sends the codebook and the numbered email, and asks the LLM to return field arrays using the numbers.


In [8]:
def build_encoded_prompt(encoded_words, codebook):
    codebook_text = "\n".join(f"{tag} = {word}" for tag, word in codebook.items())
    encoded_text = " ".join(encoded_words)
    
    return f"""Here is an email where each word has been replaced with a number.

Codebook:
{codebook_text}

Encoded email:
{encoded_text}

Extract the email header fields. Return ONLY the field lines below, nothing else.
Fill each with the numbers that represent that field's words, comma-separated inside square brackets.
Use empty brackets [] for fields not present.
Ignore government stamps. Only extract from the first email.

SENDER_NAME:
SENDER_EMAIL:
RECIPIENT_NAMES:
RECIPIENT_EMAILS:
CC_NAMES:
CC_EMAILS:
DATE:
SUBJECT:"""


prompt = build_encoded_prompt(encoded, codebook)
print(f"Prompt length: {len(prompt)} chars")
print()
print("Last 300 chars:")
print(prompt[-300:])


Prompt length: 8926 chars

Last 300 chars:
ch with the numbers that represent that field's words, comma-separated inside square brackets.
Use empty brackets [] for fields not present.
Ignore government stamps. Only extract from the first email.

SENDER_NAME:
SENDER_EMAIL:
RECIPIENT_NAMES:
RECIPIENT_EMAILS:
CC_NAMES:
CC_EMAILS:
DATE:
SUBJECT:


The LLM returns numbers. You decode them back to words using the codebook.


In [9]:
response = client.chat.completions.create(
    model="gpt-4.1-2025-04-14",
    messages=[{"role": "user", "content": prompt}],
)

raw_response = response.choices[0].message.content
print("Raw LLM response (numbers):")
print(raw_response)
print()

# Decode: swap each number back to its original word
print("Decoded:")
for line in raw_response.strip().split("\n"):
    if ": [" in line:
        key, val = line.split(": ", 1)
        # Extract numbers from the brackets
        numbers = [n.strip() for n in val.strip("[]").split(",") if n.strip()]
        decoded_words = [codebook.get(n, n) for n in numbers]
        print(f"  {key}: {' '.join(decoded_words)}")


Raw LLM response (numbers):
SENDER_NAME: [30,31]
SENDER_EMAIL: [32]
RECIPIENT_NAMES: [42,43,44,46,47,49,50,52,53,55,56,59,61,62,63,65,66,68,69,71,72,74,75,77,78,80,81]
RECIPIENT_EMAILS: [45,48,51,54,60,64,67,70,73,76,79,82]
CC_NAMES: [85,86,88,89,91,92,94,95,96,98,99,101,102,104,105,107,108,110,111,113,114,116,117]
CC_EMAILS: [87,90,93,97,100,103,106,109,112,115,118]
DATE: [35,36,37,38,39,40]
SUBJECT: [121,122,123,124,125]

Decoded:
  SENDER_NAME: Mary Hain
  SENDER_EMAIL: <mary.hain@enron.com>
  RECIPIENT_NAMES: Phillip K Allen Robert Badeer Tim Belden Shelia Benke Donald M- Black William S Bradford Rick Buy Andre Cangucu Alan Comnes Wanda Curry Jeff Dasovich Karen Denne
  RECIPIENT_EMAILS: <phillip.allen@enron.com>, <robert.badeer@enron.com>, <tim.belden@enron.com>, <shelia.benke@enron.com>, <donald.black@enron.com>, <william.bradford@enron.com>, <rick.buy@enron.com>, <andre.cangucu@enron.com>, <alan.comnes@enron.com>, <wanda.curry@enron.com>, <jeff.dasovich@enron.com>, <karen.denne@

The numbered approach produces the same result. The question is whether it uses fewer output tokens.


In [10]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4.1")

# Plain text response from section 2
plain_response = "\n".join(
    f"{key}: {values}" for key, values in result.items()
)

plain_tokens = len(enc.encode(plain_response))
dingbat_tokens = len(enc.encode(raw_response))

print("Output token comparison:")
print(f"  Plain text: {plain_tokens} tokens")
print(f"  Dingbats:   {dingbat_tokens} tokens")
print(f"  Ratio:      {plain_tokens / max(dingbat_tokens, 1):.1f}x")


Output token comparison:
  Plain text: 443 tokens
  Dingbats:   213 tokens
  Ratio:      2.1x


The numbered encoding used roughly half as many output tokens as the plain text approach. If you're processing millions of files, that adds up.

With some creative prompt engineering, you can push this further — for example, asking the LLM to output only the start and end word numbers for each field zone, and parsing the values with regex on your end.

One caveat: the encoding splits on whitespace, so punctuation stays attached to words. `kay.mann@enron.com>` (with the closing angle bracket) would decode as-is. A strip step on the decoded values handles this.

This is not the approach we'll use for this course, but it's worth remembering if you need to control costs at scale.

## 3. Clean vs noisy

The real value of LLM parsing becomes clear when you compare the same operation on clean and noisy input. 

The cell below runs both — a clean email that the template parser handled successfully, and a noisy one that it couldn't.


In [11]:
records_path = Path("data/parsed_records_enron.json")

with open(records_path) as f:
    parsed_records = json.load(f)

clean = parsed_records[1]
clean_text = (TXT_DIR / f"{clean['doc_id']}.txt").read_text(encoding="utf-8")

print("=== CLEAN EMAIL ===")
print(f"  Template result:")
print(f"    from:    {clean['from']}")
print(f"    to:      {clean['to']}")
print(f"    sent:    {clean['sent']}")
print(f"    subject: {clean['subject']}")
print(f"  LLM result:")
llm_clean = parse_with_llm(clean_text, client)
for key, values in llm_clean.items():
    print(f"    {key}: {values}")
print()

noisy_text = failure_paths[0].read_text(encoding="utf-8")
print("=== NOISY EMAIL (template failed) ===")
print(f"  Template result: FAILED — no output")
print(f"  LLM result:")
llm_noisy = parse_with_llm(noisy_text, client)
for key, values in llm_noisy.items():
    print(f"    {key}: {values}")


=== CLEAN EMAIL ===
  Template result:
    from:    EDIS Email Service <edismail@incident.com>
    to:      Motley, Matt <matt.motley@enron.com>
    sent:    Fri, 5 Apr 2002 00:07:00 -0800 (PST)
    subject: [EDIS]  EQ 4 2 SAN BERNARDINO COUNTY [News: Statewide]
  LLM result:
    SENDER_NAME: ['EDIS Email Service']
    SENDER_EMAIL: ['edismail@incident.com']
    RECIPIENT_NAMES: ['Motley, Matt']
    RECIPIENT_EMAILS: ['matt.motley@enron.com']
    CC_NAMES: []
    CC_EMAILS: []
    DATE: ['Fri, 5 Apr 2002 00:07:00 -0800 (PST)']
    SUBJECT: ['[EDIS]  EQ 4 2 SAN BERNARDINO COUNTY [News: Statewide]']

=== NOISY EMAIL (template failed) ===
  Template result: FAILED — no output
  LLM result:
    SENDER_NAME: ['Mary Hain']
    SENDER_EMAIL: ['mary.hain@enron.com']
    RECIPIENT_NAMES: ['Phillip K Allen', 'Robert Badeer', 'Tim Belden', 'Shelia Benke', 'Donald M- ECT Origination Black', 'William S Bradford', 'Rick Buy', 'Andre Cangucu', 'Alan Comnes', 'Wanda Curry', 'Jeff Dasovich', 'Karen Den

On the clean email, the LLM returns the same fields the template parser already extracted — at higher cost and lower speed. Our earlier parser did not separate emails from names, but we'll handle that separately later. Fundamentally, they're the same.

On the noisy email, the LLM succeeds where the template parser returned nothing.

The tradeoff here is that we could have made a new template to match that edge case -- and in some cases, that could genuinely be worth it. However, you will eventually reach the point where you'd make so many templates, you're essentially just rewriting a symbolic version of every individual email.

It's worth passing diverse edge cases off to an LLM once you have captured the bulk of obvious examples. 


## 4. Cost and speed

On clean text, the LLM adds cost but no value. Note: this notebook uses `gpt-4.1` for demonstration, but a smaller model like `gpt-4.1-mini` handles this structured extraction task just as well at a fraction of the cost. On noisy text, it's the only tool that works. The two are complementary — each suited to different inputs.

|  | Regex parser | LLM (gpt-4.1) |
|---|---|---|
| Speed | ~1,000/sec | ~2-5/sec |
| Cost (5,000 emails) | Free | ~$2-10 |
| Determinism | Yes | No |
| Hallucination risk | None | Low but non-zero |
| Clean layouts | Yes | Yes (same result, more expensive) |
| OCR noise | Copies faithfully | Corrects through comprehension |
| Unknown layouts | Returns None | Infers from meaning |

## 5. Batch API

You have 70 emails that the template parser couldn't handle. You could send them to the API one at a time, but most LLM providers offer a Batch API that accepts all of your requests in a single file. It runs them asynchronously at reduced cost — the trade-off is that results arrive minutes to hours later, rather than immediately.

The next cell packages each failure into a JSONL file — one request per email, using the same prompt you tested above.

In [12]:
batch_file_path = Path("data/llm_parse_batch.jsonl")

with open(batch_file_path, "w") as f:
    for txt_path in failure_paths:
        text = txt_path.read_text(encoding="utf-8")
        doc_id = txt_path.stem

        request = {
            "custom_id": doc_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4.1-2025-04-14",
                "messages": [{
                    "role": "user",
                    "content": PARSE_PROMPT.format(text=text[:2000])
                }],
            }
        }
        f.write(json.dumps(request) + "\n")

print(f"{len(failure_paths)} requests written to {batch_file_path}")


70 requests written to data/llm_parse_batch.jsonl


The next cell uploads the batch file and submits it.


In [13]:
batch_input = client.files.create(
    file=open(batch_file_path, "rb"),
    purpose="batch"
)

batch = client.batches.create(
    input_file_id=batch_input.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
)

batch_id_path = Path("data/batch_id.txt")
batch_id_path.write_text(batch.id)
print(f"Batch submitted: {batch.id}")


Batch submitted: batch_69cad93ead948190b2c318de477af56b


There are quite a number of config options available within the Batch API of each provider. We won't cover those here.

The following cell retrieves the finished batch and parses the results. If the batch hasn't completed yet, it falls back to pre-saved results from a previous run.

Feel free to skip the wait and continue, or wait for the batch to finish -- although that could take anywhere from 5 minutes to 24 hours.


In [17]:
llm_results = {}

batch_id_path = Path("data/batch_id.txt")
if batch_id_path.exists():
    batch_id = batch_id_path.read_text().strip()
    batch_status = client.batches.retrieve(batch_id)
    if batch_status.status == "completed":
        for line in client.files.content(batch_status.output_file_id).text.strip().split("\n"):
            response = json.loads(line)
            llm_results[response["custom_id"]] = parse_llm_response(
                response["response"]["body"]["choices"][0]["message"]["content"]
            )

if not llm_results:
    with open("data/llm_parse_results.jsonl") as f:
        for line in f:
            response = json.loads(line)
            llm_results[response["custom_id"]] = parse_llm_response(
                response["response"]["body"]["choices"][0]["message"]["content"]
            )

print(f"{len(llm_results)} results loaded")


70 results loaded


## 6. Batch results

Each failure case now has a parsed result. The cell below shows what the LLM extracted from each one.

In [18]:
for doc_id, result in llm_results.items():
    print(f"=== {doc_id} ===")
    for key, values in result.items():
        print(f"  {key}: {values}")
    print()


=== E076559BC ===
  SENDER_NAME: ['Mary Hain']
  SENDER_EMAIL: ['mary.hain@enron.com']
  RECIPIENT_NAMES: ['Phillip K Allen', 'Robert Badeer', 'Tim Belden', 'Shelia Benke', 'Donald M- ECT Origination Black', 'William S Bradford', 'Rick Buy', 'Andre Cangucu', 'Alan Comnes', 'Wanda Curry', 'Jeff Dasovich', 'Karen Denne']
  RECIPIENT_EMAILS: ['phillip.allen@enron.com', 'robert.badeer@enron.com', 'tim.belden@enron.com', 'shelia.benke@enron.com', 'donald.black@enron.com', 'william.bradford@enron.com', 'rick.buy@enron.com', 'andre.cangucu@enron.com', 'alan.comnes@enron.com', 'wanda.curry@enron.com', 'jeff.dasovich@enron.com', 'karen.denne@enron.com']
  CC_NAMES: ['Debra Davidson', 'Paula Warren', 'Mercy Gil', 'Karen K Heathman', 'Lysa Akin', 'Leticia Botello', 'Joseph Alamo', 'Janette Elbertson', 'Bernadette Hawkins', 'Sharon Purswell', 'Maureen McVicker']
  CC_EMAILS: ['debra.davidson@enron.com', 'paula.warren@enron.com', 'mercy.gil@enron.com', 'karen.heathman@enron.com', 'lysa.akin@enron.c

Every failure case now has structured output — sender, recipients, date, subject — extracted through comprehension rather than pattern matching. These are the cases where OCR corruption made the headers unrecognisable to the template parser.

Spot-check one result against the raw text to verify the LLM got it right:

In [ ]:
# Spot-check: compare LLM output to raw text for the first failure
first_id = list(llm_results.keys())[0]
raw_text = (TXT_DIR / f"{first_id}.txt").read_text(encoding="utf-8")

print(f"=== Raw text ({first_id}, first 400 chars) ===")
print(raw_text[:400])
print()
print(f"=== LLM extracted ===")
for key, values in llm_results[first_id].items():
    print(f"  {key}: {values}")

## 7. Merge and save

The template parser handled 4,841 emails. The LLM handled the remaining 70. The cell below loads the template results, adds the LLM results, and writes the combined dataset.

In [19]:
def first_or_empty(arr):
    """Safely get the first element of an array, or empty string."""
    return arr[0] if arr else ""


with open("data/parsed_records_enron.json") as f:
    all_records = json.load(f)

for doc_id, result in llm_results.items():
    all_records.append({
        "doc_id": doc_id,
        "from": first_or_empty(result.get("SENDER_NAME", [])),
        "from_email": first_or_empty(result.get("SENDER_EMAIL", [])),
        "sent": first_or_empty(result.get("DATE", [])),
        "to": result.get("RECIPIENT_NAMES", []),
        "to_emails": result.get("RECIPIENT_EMAILS", []),
        "cc": result.get("CC_NAMES", []),
        "cc_emails": result.get("CC_EMAILS", []),
        "subject": first_or_empty(result.get("SUBJECT", [])),
        "_template": "llm",
    })

with open("data/all_parsed_records.json", "w") as f:
    json.dump(all_records, f, indent=2, ensure_ascii=False)

print(f"{len(all_records)} total records ({len(all_records) - len(llm_results)} template + {len(llm_results)} LLM)")

4911 total records (4841 template + 70 LLM)


The combined dataset — 4,841 template-parsed records plus 70 LLM-parsed records — is saved to `data/all_parsed_records.json`. Every email in the corpus now has a parsed result.

## Summary

- A prompt that shows the LLM exactly what correct output looks like — with a concrete example — produces more reliable results than verbose instructions
- Asking the LLM to return flat arrays (one per field) keeps the response predictable and easy to parse
- On clean text, the LLM and the template parser produce the same result — the LLM adds cost with no benefit
- On noisy text, the LLM corrects OCR artifacts through comprehension — this is where it earns its keep
- The Batch API processes failures in bulk at reduced cost compared to individual calls
- The combined dataset (template + LLM) is saved to `data/all_parsed_records.json`
